In [ ]:
from pathlib import Path
import numpy as np 
import tifffile as tf
import pandas as pd
import skimage
from sklearn.preprocessing import MinMaxScaler
import pickle
import scipy.cluster.hierarchy as sch 
import matplotlib.pyplot as plt
from collections import Counter
import cv2
from matplotlib_scalebar.scalebar import ScaleBar
from pyFlowSOM import map_data_to_nodes, som
from alpineer import io_utils, misc_utils
from pyarrow.lib import ArrowInvalid
!pip install parc
import parc
import time 
from tqdm import tqdm
!pip install tqdm_joblib
from tqdm_joblib import tqdm_joblib
from joblib import Parallel, delayed

In [ ]:
def compute_som_weights(pixels_subset_no_meta, prefix, xdim = 10, ydim = 10, num_passes = 1, lr_start=0.05, lr_end=0.01, seed = 42):
    som_weights = som(
            data=pixels_subset_no_meta.values.astype(np.float64), xdim=xdim, ydim=ydim,
            rlen=num_passes, alpha_range=(lr_start, lr_end), seed=seed
        )

    som_weights = np.reshape(som_weights, (xdim * ydim, som_weights.shape[-1]))
    weights_df = pd.DataFrame(som_weights, columns=pixels_subset_no_meta.columns.values)
    return weights_df

def assign_som_clusters(weights_df, pixels_no_meta, batch_size): 
    cluster_labels = []
    
    for i in np.arange(0, pixels_no_meta.shape[0], batch_size):
        cluster_labels.append(map_data_to_nodes(
            weights_df.values.astype(np.float64),
            pixels_no_meta.loc[
                i:min(i + batch_size-1, pixels_no_meta.shape[0]), pixels_no_meta.columns.values
            ].values.astype(np.float64)
        )[0])
    pixels_no_meta['pixel_som_cluster'] = np.concatenate(cluster_labels)
    return pixels_no_meta
    
    

def cluster_som(pixels_subset, pixels, prefix, batch_size, drop_position, weights_df = None, xdim = None, ydim= None):
    if(drop_position):
        pixels_subset_no_meta = pixels_subset.drop(columns=['X', 'Y'])
        pixels_no_meta = pixels.drop(columns=['X', 'Y'])
    else:
        pixels_subset_no_meta = pixels_subset
        pixels_no_meta = pixels
    if(weights_df is None):
        if(xdim is None and ydim is None):
            weights_df = compute_som_weights(pixels_subset_no_meta, prefix)
        else:
            weights_df = compute_som_weights(pixels_subset_no_meta, prefix, xdim=xdim, ydim=ydim)
    pixels_no_meta_som_clusters = assign_som_clusters(weights_df, pixels_no_meta, batch_size)
   

    return pixels_no_meta_som_clusters, weights_df

In [ ]:
def cluster_parc(pixels):
    parc1 = parc.PARC(pixels.values, jac_weighted_edges = False, random_seed=0, knn=50, small_pop=500)
    parc1.run_PARC() 
    parc_labels = parc1.labels
    pixels['label'] = parc_labels
    
    pixels_grouped_by_labels = pixels.groupby("label").mean()

    d = sch.distance.pdist(pixels_grouped_by_labels, metric='cosine')
    L = sch.linkage(d, method="average")

    tol = 0.1
    clusters = sch.fcluster(L, tol * d.max(), "distance")-1
    label2cluster = dict(zip(list(set(pixels.label)), clusters))
    labels = np.array(list(map(label2cluster.get, pixels.label)))
    pixels['cluster'] = labels

    return pixels 

In [ ]:
key  = "" # sample identifier
cell_clustering_path = r"" # path to cell phenotyping results
pixel_clustering_path = r"" # path to pixel clustering results

In [ ]:
non_empty_pixels = pd.read_csv(pixel_clustering_path + '/non_empty_pixels.csv').drop(columns=['Unnamed: 0'])

In [ ]:
region_size = 1024

In [ ]:
non_empty_pixels_grouped = non_empty_pixels.copy()

In [ ]:
non_empty_pixels_grouped["region_X"] = non_empty_pixels_grouped["X"] // region_size
non_empty_pixels_grouped["region_Y"] = non_empty_pixels_grouped["Y"] // region_size

In [ ]:
non_empty_pixels_grouped = non_empty_pixels_grouped.groupby(["region_X", "region_Y"])

In [ ]:
del non_empty_pixels

In [ ]:
blocks = {
    (br, bc): block_df
    for (br, bc), block_df in non_empty_pixels_grouped
}

In [ ]:
blocks_processed = {}
for (br, bc), block_df in blocks.items():
    if(block_df.shape[0] > 100000):
        blocks_processed[(br, bc)] = block_df

In [ ]:
del non_empty_pixels_grouped
del blocks

In [ ]:
clustering_time_dict = {}
clustering_time_dict['PARC'] = {}
clustering_time_dict['SOM'] = {}

clustering_time_dict['PARC'][region_size]=[]
clustering_time_dict['SOM'][region_size]=[]

In [ ]:
def run_parc_som(region_pixels, weights_df=None):
    region_pixels_no_meta = region_pixels.drop(columns=['region_X', 'region_Y', 'X', 'Y']).reset_index().drop(columns=['index'])
    scaler = MinMaxScaler()
    x_scaled = scaler.fit_transform(region_pixels_no_meta)
    region_pixels_no_meta_scaled = pd.DataFrame(x_scaled, columns=region_pixels_no_meta.columns)

    start = time.perf_counter()
    pixels_no_meta_som_clusters, weights_df = cluster_som(region_pixels_no_meta_scaled, region_pixels_no_meta_scaled, '', 100000, drop_position=False, weights_df=weights_df, xdim=4, ydim=4)
    end = time.perf_counter()
    som_time = end - start

    region_pixels_no_meta = region_pixels.drop(columns=['region_X', 'region_Y', 'X', 'Y']).reset_index().drop(columns=['index'])
    scaler = MinMaxScaler()
    x_scaled = scaler.fit_transform(region_pixels_no_meta)
    region_pixels_no_meta_scaled = pd.DataFrame(x_scaled, columns=region_pixels_no_meta.columns)

    start = time.perf_counter()
    pixels_no_meta_parc_clusters = cluster_parc(region_pixels_no_meta_scaled)
    end = time.perf_counter()
    parc_time = end - start
    
    del region_pixels_no_meta
    del region_pixels_no_meta_scaled
    del pixels_no_meta_som_clusters
    del pixels_no_meta_parc_clusters

    return (som_time, parc_time)

In [ ]:
with tqdm_joblib(tqdm(desc="Clustering image regions", total=len(blocks_processed))) as progress_bar:
    result = Parallel(n_jobs=15)(delayed(run_parc_som)(region_pixels, None) for (br, bc), region_pixels in blocks_processed.items())

In [ ]:
som_times, parc_times = zip(*result)

clustering_time_dict['SOM'][region_size] = list(som_times)
clustering_time_dict['PARC'][region_size] = list(parc_times)

In [ ]:
with open(pixel_clustering_path + '/run_time.pickle', 'wb') as f: 
    pickle.dump(clustering_time_dict, f)

In [ ]:
som_times = clustering_time_dict['SOM'][1024]
parc_times = clustering_time_dict['PARC'][1024]

In [ ]:
print(len(som_times))

In [ ]:
from scipy import stats

In [ ]:
import matplotlib as mpl

In [ ]:
mpl.rcParams["font.size"] = 15
som = np.array(list(som_times))
parc = np.array(list(parc_times))

_, p_value = stats.ttest_rel(som, parc)

if p_value < 0.001:
    sig_text = '***'
elif p_value < 0.01:
    sig_text = '**'
elif p_value < 0.05:
    sig_text = '*'
else:
    sig_text = 'n.s.'

fig, ax = plt.subplots(figsize=(6, 4))

y_pos = [0, 0.4]

bp = ax.boxplot(
    [som, parc],
    vert=False,
    positions=y_pos,
    widths=0.25,
    patch_artist=True,
    showfliers=True
)

colors = ['dodgerblue', 'red']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for element in ['whiskers', 'caps', 'medians']:
    for item in bp[element]:
        item.set_color('black')

ax.set_xscale('log')
ax.set_yticks(y_pos)
ax.set_yticklabels(['SOM', 'PARC'])
ax.set_xlabel('Time (s)')
ax.set_ylabel('Pixel Clustering Algorithms')
ax.set_title('Run Time')

x_max = max(som.max(), parc.max())
factor = 1.4

line_x = x_max * factor
text_x = line_x * 1.07

ax.plot([line_x, line_x], y_pos, color='black', lw=1.5)
ax.text(
    text_x, np.mean(y_pos), sig_text,
    va='center', ha='left',
    fontsize=12, rotation=90
)

plt.tight_layout()
plt.show()


In [ ]:
keys = ['/Multiplex 2/ANCA 9', 
        '/Multiplex 2/CTRL 7', 
        '/Multiplex 6/MP3-058-2',
        '/Multiplex 6/L554-5',
        '/Multiplex 6/CTRL 4', 
        '/Multiplex 8/Med 2', 
        '/Multiplex 9/L554-4', 
        '/Multiplex 12/Rqo20']

In [ ]:
run_time_all_samples = {}

In [ ]:
pixel_clustering_base = 'Y:/coskun-lab/Aditi/01/Results/pixel_clustering'

In [ ]:
for key in keys: 
    with open(pixel_clustering_base + key + '/run_time.pickle', 'rb') as f: 
        run_time_all_samples[key] = pickle.load(f)

In [ ]:
run_time_combined = {}

for sample in run_time_all_samples.values():
    for key, value in sample.items():
        run_time_combined.setdefault(key, []).extend(value[1024])

In [ ]:
som_times_combined = run_time_combined['SOM']
parc_times_combined = run_time_combined['PARC']

In [ ]:
len(som_times_combined)

In [ ]:
import matplotlib as mpl
from scipy import stats 

mpl.rcParams["font.size"] = 15
som = np.array(list(som_times_combined))
parc = np.array(list(parc_times_combined))

_, p_value = stats.ttest_rel(som, parc)

if p_value < 0.001:
    sig_text = '***'
elif p_value < 0.01:
    sig_text = '**'
elif p_value < 0.05:
    sig_text = '*'
else:
    sig_text = 'n.s.'

fig, ax = plt.subplots(figsize=(6, 4))

y_pos = [0, 0.4]

bp = ax.boxplot(
    [som, parc],
    vert=False,
    positions=y_pos,
    widths=0.25,
    patch_artist=True,
    showfliers=True
)

colors = ['dodgerblue', 'red']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

for element in ['whiskers', 'caps', 'medians']:
    for item in bp[element]:
        item.set_color('black')

# Axis settings
ax.set_xscale('log')
ax.set_yticks(y_pos)
ax.set_yticklabels(['SOM', 'PARC'])
ax.set_xlabel('Time (s)')
ax.set_ylabel('Pixel Clustering Algorithms')
ax.set_title('Run Time')

x_max = max(som.max(), parc.max())
factor = 1.4

line_x = x_max * factor
text_x = line_x * 1.07

ax.plot([line_x, line_x], y_pos, color='black', lw=1.5)
ax.text(
    text_x, np.mean(y_pos), sig_text,
    va='center', ha='left',
    fontsize=12, rotation=90
)

plt.tight_layout()
plt.show()
